# Beautiful Soup 3 — From the S&P 500 list to two Yahoo Finance quotes

**Goal:** scrape Wikipedia's S&P 500 constituents table, choose **exactly two** companies, then iterate over them and extract a quote summary and its visible quote statistics from Yahoo Finance.

In [1]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

WIKIPEDIA_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
YAHOO_URL = "https://finance.yahoo.com/quote/{symbol}/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/157.0.0.0 Safari/537.36",   # a real Chrome User-Agent
    "Accept-Language": "en-US,en;q=0.9",
}
session = requests.Session()
session.headers.update(HEADERS)

## 1. Turn the Wikipedia table into a DataFrame

This repeats the Beautiful Soup technique from notebook 2 so this notebook is independently runnable.

In [2]:
wiki_response = session.get(WIKIPEDIA_URL, timeout=20)
wiki_response.raise_for_status()
wiki_soup = BeautifulSoup(wiki_response.text, "html.parser")

table = wiki_soup.select_one("table#constituents")
if table is None:
    raise RuntimeError("Could not find table#constituents. Inspect Wikipedia's current HTML and update the selector.")

header_row = table.find("tr")
columns = [cell.get_text(" ", strip=True) for cell in header_row.find_all("th")]
rows = []
for row in table.select("tr")[1:]:
    cells = row.find_all("td")
    if len(cells) == len(columns):
        rows.append([cell.get_text(" ", strip=True) for cell in cells])

sp500 = pd.DataFrame(rows, columns=columns)
sp500.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,0000066740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee , Wisconsin",2017-07-26,0000091142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,0000001800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,0001551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin , Ireland",2011-07-06,0001467373,1989


## 2. Select two companies only

`head(2)` establishes a hard teaching limit. We retain both the source symbol and a Yahoo-formatted symbol because share-class tickers use `-` on Yahoo Finance.

In [3]:
companies = sp500.loc[:, ["Symbol", "Security"]].head(2).copy()
companies["yahoo_symbol"] = companies["Symbol"].str.replace(".", "-", regex=False)
companies

,Symbol,Security,yahoo_symbol
0,MMM,3M,MMM
1,AOS,A. O. Smith,AOS


## 3. Parse a Yahoo quote header

The earlier `fin-streamer` selectors with `regularMarketPrice`, `regularMarketChange`, and `regularMarketChangePercent` do not match the current quote-header markup. The current header uses `<span>` tags with `data-testid` values `qsp-price`, `qsp-price-change`, and `qsp-price-change-percent`.

Browser DevTools **Elements** shows the browser-live DOM, while Beautiful Soup parses the raw `response.text` downloaded by `requests`. If a selector returns `None`, check for that exact attribute in `response.text`; the response may not contain the element, or the selector may no longer match.

In [4]:
def text_or_none(tag):
    return tag.get_text(" ", strip=True) if tag else None


def parse_yahoo_quote(html, source_symbol, yahoo_symbol, fallback_company):
    soup = BeautifulSoup(html, "html.parser")
    heading = soup.select_one('[data-testid="quote-header"] h1') or soup.find("h1")
    company = text_or_none(heading) or fallback_company

    quote = {
        "source_symbol": source_symbol,
        "yahoo_symbol": yahoo_symbol,
        "company": company,
        "regular_price": text_or_none(soup.select_one('span[data-testid="qsp-price"]')),
        "price_change": text_or_none(soup.select_one('span[data-testid="qsp-price-change"]')),
        "percent_change": text_or_none(soup.select_one('span[data-testid="qsp-price-change-percent"]')),
    }
    return quote, soup, company

## 4. Parse the visible quote-statistics grid

The grid below the header is a list of local `<li>` rows rather than an HTML table. We locate known quote-statistic values, move to each row, and read its label and value without using generated CSS classes. Rows that are absent from `response.text` are skipped. This covers usable rows such as Previous Close, Open, Bid, Ask, ranges, volume, market cap, beta, P/E, EPS, earnings, dividend/yield, target estimate, and optional Morningstar Rating.

In [5]:
STATISTIC_FIELDS = [
    "regularMarketPreviousClose", "regularMarketOpen", "bid", "ask",
    "regularMarketDayRange", "fiftyTwoWeekRange", "regularMarketVolume",
    "averageDailyVolume3Month", "marketCap", "beta", "trailingPE",
    "epsTrailingTwelveMonths", "earningsTimestamp", "dividendRate",
    "dividendYield", "trailingAnnualDividendRate",
    "trailingAnnualDividendYield", "targetMeanPrice",
]
MISSING_VALUES = {"", "-", "--", "N/A", "n/a"}


def parse_quote_statistics(soup, company, source_symbol, yahoo_symbol):
    statistic_rows = []
    for value_tag in soup.find_all(attrs={"data-field": STATISTIC_FIELDS}):
        row = value_tag.find_parent("li")
        if row is not None and row not in statistic_rows:
            statistic_rows.append(row)

    morningstar_label = soup.find(
        string=lambda text: text and text.strip() == "Morningstar Rating"
    )
    morningstar_row = morningstar_label.find_parent("li") if morningstar_label else None
    if morningstar_row is not None and morningstar_row not in statistic_rows:
        statistic_rows.append(morningstar_row)

    records = []
    for row in statistic_rows:
        label_tag = row.find(["span", "a"], attrs={"title": True}) or row.find(["span", "a"])
        value_tag = label_tag.find_next_sibling() if label_tag else None
        value_tag = value_tag or row.find(attrs={"data-field": True})
        label = text_or_none(label_tag)
        value = text_or_none(value_tag)

        if value is None and label:
            value = " ".join(text for text in row.stripped_strings if text != label)

        if label and value and value not in MISSING_VALUES:
            records.append({
                "company": company,
                "source_symbol": source_symbol,
                "yahoo_symbol": yahoo_symbol,
                "statistic": label,
                "value": value,
            })
    return records

## 5. Iterate over the two selected companies

The loop keeps the request limit at exactly two. It records a failed quote without stopping the lesson, and adds only statistics that were usable in that response.

In [6]:
if len(companies) != 2:
    raise RuntimeError("Expected exactly two companies from the S&P 500 table.")

quote_records = []
statistic_records = []

for company in companies.itertuples(index=False):
    url = YAHOO_URL.format(symbol=company.yahoo_symbol)
    try:
        response = session.get(url, timeout=20)
        response.raise_for_status()
        quote, soup, company_name = parse_yahoo_quote(
            response.text, company.Symbol, company.yahoo_symbol, company.Security
        )
        quote_records.append(quote)
        statistic_records.extend(
            parse_quote_statistics(soup, company_name, company.Symbol, company.yahoo_symbol)
        )
        print(f"Fetched {company.yahoo_symbol}: HTTP {response.status_code}")
    except requests.RequestException as error:
        quote_records.append({
            "source_symbol": company.Symbol,
            "yahoo_symbol": company.yahoo_symbol,
            "company": company.Security,
            "regular_price": None,
            "price_change": None,
            "percent_change": None,
            "error": str(error),
        })
        print(f"Could not fetch {company.yahoo_symbol}: {error}")
    time.sleep(1.5)

Fetched MMM: HTTP 200
Fetched AOS: HTTP 200


## 6. Headline quote output

`quotes` has one headline-price record for each of the two selected companies.

In [7]:
quotes = pd.DataFrame(quote_records)
quotes

,source_symbol,yahoo_symbol,company,regular_price,price_change,percent_change
0,MMM,MMM,3M Company (MMM),186.31,-1.46,(-0.78%)
1,AOS,AOS,A. O. Smith Corporation (AOS),64.86,-1.71,(-2.58%)


## 7. Quote-statistics output

`quote_statistics` has one row for every usable statistics-grid label/value pair returned for each of the two companies.

In [8]:
quote_statistics = pd.DataFrame(
    statistic_records,
    columns=["company", "source_symbol", "yahoo_symbol", "statistic", "value"],
)
quote_statistics

,company,source_symbol,yahoo_symbol,statistic,value
0,3M Company (MMM),MMM,MMM,Previous Close,187.77
1,3M Company (MMM),MMM,MMM,Open,184.04
2,3M Company (MMM),MMM,MMM,Day's Range,183.60 - 187.59
3,3M Company (MMM),MMM,MMM,52 Week Range,139.34 - 196.90
4,3M Company (MMM),MMM,MMM,Volume,"1,323,236"
5,3M Company (MMM),MMM,MMM,Market Cap (intraday),96.272B
6,3M Company (MMM),MMM,MMM,PE Ratio (TTM),27.81
7,3M Company (MMM),MMM,MMM,EPS (TTM),6.70
8,3M Company (MMM),MMM,MMM,1y Target Est,192.07
9,3M Company (MMM),MMM,MMM,Morningstar Rating,Rating available with Stock Picks access


## Reflection

This is a small ETL pipeline: **extract** two symbols from Wikipedia, **transform** their ticker format, then **load** headline quotes and quote-statistics records into DataFrames. Try changing only the explicit two-company selection—not the request rate or loop limit—and compare the returned records.